## 1. Prompt chaining (sequential workflow)

The simplest pattern: run step A, feed its output into step B, feed that
into step C. Each step is a small, focused (stubbed) LLM call.


In [1]:
"""
Problem statement: Find the riskiest clauses from the contract.
"""

def step_extract_clauses(contract_text: str) -> list:
    """Step A. # TODO: replace with a real LLM call that extracts clauses."""
    # Split text on full stop for this notebook.
    return [c.strip() for c in contract_text.split(".") if c.strip()]

def step_classify_risk(clause: str) -> str:
    """Step B. # TODO: replace with a real LLM call that classifies risk."""
    risky_words = ["penalty", "terminate", "non-refundable", "liable"]
    print("high-risk" if any(w in clause.lower() for w in risky_words) else "low-risk")
    return "high-risk" if any(w in clause.lower() for w in risky_words) else "low-risk"

def step_summarize_flagged(flagged_clauses: list) -> str:
    """Step C. # TODO: replace with a real LLM call that writes a summary."""
    if not flagged_clauses:
        return "No high-risk clauses found."
    return "High-risk clauses found: " + "; ".join(flagged_clauses)

def sequential_workflow(contract_text: str) -> str:
    clauses = step_extract_clauses(contract_text)                       # Step A
    classified = [(c, step_classify_risk(c)) for c in clauses]          # Step B
    print(f"classified: {classified}")
    flagged = [c for c, risk in classified if risk == "high-risk"]      # filter
    print(f"flagged: {flagged}")
    summary = step_summarize_flagged(flagged)                           # Step C
    return summary

contract = (
    "The vendor shall deliver goods within 30 days. "
    "Late delivery incurs a penalty of 2% per week. "
    "All payments are non-refundable once processed. "
    "Either party may request a status update at any time."
)

print(sequential_workflow(contract))


low-risk
high-risk
high-risk
low-risk
classified: [('The vendor shall deliver goods within 30 days', 'low-risk'), ('Late delivery incurs a penalty of 2% per week', 'high-risk'), ('All payments are non-refundable once processed', 'high-risk'), ('Either party may request a status update at any time', 'low-risk')]
flagged: ['Late delivery incurs a penalty of 2% per week', 'All payments are non-refundable once processed']
High-risk clauses found: Late delivery incurs a penalty of 2% per week; All payments are non-refundable once processed


**Exercise 3.1:** Print out `classified` (not just the final summary) so you
can see every clause's risk label. Add one more risky keyword of your choice
to `step_classify_risk` and confirm it changes the output on a new example
contract you write yourself.


In [3]:
"""
Problem statement: Find the riskiest clauses from the contract.
"""

def step_extract_clauses(contract_text: str) -> list:
    """Step A. Extract clauses."""
    return [c.strip() for c in contract_text.split(".") if c.strip()]


def step_classify_risk(clause: str) -> str:
    """Step B. Classify clause as high-risk or low-risk."""
    risky_words = ["penalty", "terminate", "non-refundable", "liable", "breach"]

    risk = "high-risk" if any(w in clause.lower() for w in risky_words) else "low-risk"

    print(risk)
    return risk


def step_summarize_flagged(flagged_clauses: list) -> str:
    """Step C. Summarize flagged clauses."""
    if not flagged_clauses:
        return "No high-risk clauses found."

    return "High-risk clauses found: " + "; ".join(flagged_clauses)


def sequential_workflow(contract_text: str) -> str:
    clauses = step_extract_clauses(contract_text)

    classified = [(c, step_classify_risk(c)) for c in clauses]

    print(f"classified: {classified}")

    flagged = [c for c, risk in classified if risk == "high-risk"]

    print(f"flagged: {flagged}")

    summary = step_summarize_flagged(flagged)

    return summary


# Original contract
contract = (
    "The vendor shall deliver goods within 30 days. "
    "Late delivery incurs a penalty of 2% per week. "
    "All payments are non-refundable once processed. "
    "Either party may request a status update at any time."
)

print(sequential_workflow(contract))


# New example contract to test the added keyword "breach"
new_contract = (
    "The customer shall pay the invoice within 15 days. "
    "A breach of confidentiality may result in termination of the agreement. "
    "The customer may request a copy of the invoice."
)

print("\nNew contract:")
print(sequential_workflow(new_contract))

low-risk
high-risk
high-risk
low-risk
classified: [('The vendor shall deliver goods within 30 days', 'low-risk'), ('Late delivery incurs a penalty of 2% per week', 'high-risk'), ('All payments are non-refundable once processed', 'high-risk'), ('Either party may request a status update at any time', 'low-risk')]
flagged: ['Late delivery incurs a penalty of 2% per week', 'All payments are non-refundable once processed']
High-risk clauses found: Late delivery incurs a penalty of 2% per week; All payments are non-refundable once processed

New contract:
low-risk
high-risk
low-risk
classified: [('The customer shall pay the invoice within 15 days', 'low-risk'), ('A breach of confidentiality may result in termination of the agreement', 'high-risk'), ('The customer may request a copy of the invoice', 'low-risk')]
flagged: ['A breach of confidentiality may result in termination of the agreement']
High-risk clauses found: A breach of confidentiality may result in termination of the agreement


## 2. Routing (conditional workflow)

Instead of a fixed sequence, first classify the input, then send it down one
of several different specialized paths based on that classification.


In [2]:
"""
Problem Statement: Classify the type of ticket raised by customer in a  ticketing system and soute to the respective branch to resolve the ticket.
Note: You can think ticket as a complain by the customer.
"""
def classify_ticket(ticket_text: str) -> str:
    """# TODO: replace with a real LLM call for classification."""
    text = ticket_text.lower()
    if any(w in text for w in ["refund", "charge", "invoice", "payment"]):
        return "billing"
    if any(w in text for w in ["error", "bug", "crash", "not working"]):
        return "technical"
    return "general"

def handle_billing(ticket_text: str) -> str:
    return f"[Billing team] Reviewing payment issue: '{ticket_text}'"

def handle_technical(ticket_text: str) -> str:
    return f"[Technical team] Reproducing reported issue: '{ticket_text}'"

def handle_general(ticket_text: str) -> str:
    return f"[General support] Logging inquiry: '{ticket_text}'"

def routing_workflow(ticket_text: str) -> str:
    category = classify_ticket(ticket_text)             # classify first
    if category == "billing":                           # then route
        return handle_billing(ticket_text)
    elif category == "technical":
        return handle_technical(ticket_text)
    else:
        return handle_general(ticket_text)

for ticket in [
    "I was charged twice for my last order.",
    "The app crashes every time I open the settings page.",
    "What are your business hours?",
]:
    print(routing_workflow(ticket))


[Billing team] Reviewing payment issue: 'I was charged twice for my last order.'
[Technical team] Reproducing reported issue: 'The app crashes every time I open the settings page.'
[General support] Logging inquiry: 'What are your business hours?'


**Exercise 3.2:** This is still *not* an agent even
though it branches. Explain in one sentence why not, referencing the
workflows-vs-agents.


Because it follows a fixed, predefined workflow with hard-coded rules and branches, rather than an agent that can autonomously decide what actions to take, use tools, and adapt its workflow toward a goal.

## 3. Parallelization

Run several independent checks on the same input at once (conceptually --
Python's `threading`/`asyncio` would run these concurrently in a real
system; here we call them one after another to keep the demo simple, since
the *pattern*, not real concurrency, is the teaching point).


In [4]:
"""
Execution of parallel tools
Given a text check grammer of the text, check tome and length of the text.
"""

def check_grammar(text: str) -> dict:
    """# TODO: replace with a real LLM call."""
    return {"check": "grammar", "issues_found": text.count("  ")}  # e.g. double spaces

def check_tone(text: str) -> dict:
    """# TODO: replace with a real LLM call."""
    return {"check": "tone", "flagged": "urgent" in text.lower() or "asap" in text.lower()}

def check_length(text: str) -> dict:
    """# TODO: replace with a real LLM call, or just deterministic logic like this."""
    return {"check": "length", "word_count": len(text.split())}

def parallelization_workflow(text: str) -> list:
    # In a real system these would run concurrently; the pattern (independent
    # checks combined afterward) is what matters here, not literal parallel execution.
    checks = [check_grammar, check_tone, check_length]
    return [check(text) for check in checks]

results = parallelization_workflow("This is  urgent, please respond ASAP.")
for r in results:
    print(r)


{'check': 'grammar', 'issues_found': 1}
{'check': 'tone', 'flagged': True}
{'check': 'length', 'word_count': 6}


**Exercise 3.3:** Add a fourth independent check of your own (e.g.,
`check_has_greeting`), add it to the `checks` list, and
confirm it runs alongside the others without needing to change any other
check's code. This independence is exactly why parallelization is a useful
pattern: each check is decoupled from the others.


In [5]:
"""
Execution of parallel tools

Given a text check grammar of the text, check tone and length of the text.
"""

def check_grammar(text: str) -> dict:
    """# TODO: replace with a real LLM call."""
    return {"check": "grammar", "issues_found": text.count("  ")}


def check_tone(text: str) -> dict:
    """# TODO: replace with a real LLM call."""
    return {
        "check": "tone",
        "flagged": "urgent" in text.lower() or "asap" in text.lower()
    }


def check_length(text: str) -> dict:
    """# TODO: replace with a real LLM call."""
    return {"check": "length", "word_count": len(text.split())}


# Fourth independent check
def check_has_greeting(text: str) -> dict:
    """Check whether the text contains a greeting."""
    greetings = ["hello", "hi", "hey"]
    has_greeting = any(word in text.lower() for word in greetings)

    return {"check": "greeting", "has_greeting": has_greeting}


def parallelization_workflow(text: str) -> list:
    checks = [
        check_grammar,
        check_tone,
        check_length,
        check_has_greeting
    ]

    return [check(text) for check in checks]


results = parallelization_workflow(
    "Hello, this is urgent, please respond ASAP."
)

for r in results:
    print(r)

{'check': 'grammar', 'issues_found': 0}
{'check': 'tone', 'flagged': True}
{'check': 'length', 'word_count': 7}
{'check': 'greeting', 'has_greeting': True}


## 4. Orchestrator–worker

One step plans/breaks the task into sub-tasks and delegates each to a
separate ("worker") call, then combines their outputs. This is task
decomposition applied as a workflow pattern.


In [6]:
"""
This is just to show how orchestration flows work. We will discuss this in next session.
"""

def orchestrator_plan(task: str) -> list:
    """# TODO: replace with a real LLM call that decomposes `task` into sub-tasks."""
    if "trip" in task.lower():
        return ["find flights", "find hotels", "plan itinerary", "estimate budget"]
    return ["research topic", "draft response"]

def worker_execute(subtask: str) -> str:
    """# TODO: replace with a real LLM call (or a tool call) per sub-task."""
    return f"(stub result for: '{subtask}')"

def orchestrator_worker_workflow(task: str) -> dict:
    subtasks = orchestrator_plan(task)                                 # orchestrator
    results = {st: worker_execute(st) for st in subtasks}              # workers
    combined = " | ".join(results.values())                            # combine
    return {"task": task, "subtasks": subtasks, "combined_result": combined}

import pprint
pprint.pprint(orchestrator_worker_workflow("Plan a 3-day trip to Andaman"))


{'combined_result': "(stub result for: 'find flights') | (stub result for: "
                    "'find hotels') | (stub result for: 'plan itinerary') | "
                    "(stub result for: 'estimate budget')",
 'subtasks': ['find flights',
              'find hotels',
              'plan itinerary',
              'estimate budget'],
 'task': 'Plan a 3-day trip to Andaman'}


## 5. Evaluator–optimizer

Generate an attempt, evaluate/critique it, and loop until the evaluator is
satisfied (or a max number of attempts is reached, to avoid an infinite
loop). This is a workflow-level preview of the self-reflection.


In [7]:
def generate_attempt(task: str, attempt_number: int) -> str:
    """# TODO: replace with a real LLM call."""
    # Toy stand-in: pretend quality improves with each attempt.
    quality_words = ["draft", "revised", "polished"]
    word = quality_words[min(attempt_number, len(quality_words) - 1)]
    return f"({word} attempt #{attempt_number + 1} at: {task})"

def evaluate_attempt(attempt: str) -> bool:
    """# TODO: replace with a real LLM call that judges the attempt.
    Returns True if the attempt is good enough to stop."""
    return "polished" in attempt

def evaluator_optimizer_workflow(task: str, max_attempts: int = 5) -> str:
    for i in range(max_attempts):
        attempt = generate_attempt(task, i)
        print(f"Attempt {i + 1}: {attempt}")
        if evaluate_attempt(attempt):
            print("Evaluator: satisfied, stopping.")
            return attempt
        print("Evaluator: not good enough yet, retrying...")
    print("Evaluator: max attempts reached, returning last attempt.")
    return attempt

evaluator_optimizer_workflow("write a product tagline")


Attempt 1: (draft attempt #1 at: write a product tagline)
Evaluator: not good enough yet, retrying...
Attempt 2: (revised attempt #2 at: write a product tagline)
Evaluator: not good enough yet, retrying...
Attempt 3: (polished attempt #3 at: write a product tagline)
Evaluator: satisfied, stopping.


'(polished attempt #3 at: write a product tagline)'

**Exercise 3.4:** Automation for Medical Health Insurance

In [8]:
def generate_attempt(claim: str, attempt_number: int) -> str:
    """Generate an improved claim review."""

    quality_words = ["initial review", "revised review", "complete review"]
    word = quality_words[min(attempt_number, len(quality_words) - 1)]

    return f"({word} #{attempt_number + 1}: {claim})"


def evaluate_attempt(attempt: str) -> bool:
    """Evaluate whether the claim review is complete."""

    return "complete review" in attempt


def evaluator_optimizer_workflow(
    claim: str, max_attempts: int = 5
) -> str:

    for i in range(max_attempts):

        attempt = generate_attempt(claim, i)

        print(f"Attempt {i + 1}: {attempt}")

        if evaluate_attempt(attempt):
            print("Evaluator: claim review is satisfactory, stopping.")
            return attempt

        print("Evaluator: review needs improvement, retrying...")

    print("Evaluator: maximum attempts reached.")
    return attempt


claim = (
    "Patient received treatment for a covered medical condition. "
    "Hospital bill and treatment details were submitted."
)

evaluator_optimizer_workflow(claim)

Attempt 1: (initial review #1: Patient received treatment for a covered medical condition. Hospital bill and treatment details were submitted.)
Evaluator: review needs improvement, retrying...
Attempt 2: (revised review #2: Patient received treatment for a covered medical condition. Hospital bill and treatment details were submitted.)
Evaluator: review needs improvement, retrying...
Attempt 3: (complete review #3: Patient received treatment for a covered medical condition. Hospital bill and treatment details were submitted.)
Evaluator: claim review is satisfactory, stopping.


'(complete review #3: Patient received treatment for a covered medical condition. Hospital bill and treatment details were submitted.)'